In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv("personal-armoury.csv")

print(df.head())
print(df.info())
print(df.describe())

             Name           Type  Amount  Unit Value
0      VOL-GA K12  Placed Weapon      96       12000
1   HMG-3 WINGATE  Placed Weapon     122       12000
2  M2A-304 MORTAR  Placed Weapon     240        9000
3      ZHIZDRA-45  Placed Weapon      32       20000
4   M276 AA G-GUN  Placed Weapon      39       20000
<class 'pandas.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Name        23 non-null     str  
 1   Type        23 non-null     str  
 2   Amount      23 non-null     int64
 3   Unit Value  23 non-null     int64
dtypes: int64(2), str(2)
memory usage: 1.3 KB
None
           Amount     Unit Value
count   23.000000      23.000000
mean    42.739130   74478.695652
std     51.035832  150149.357408
min      0.000000      10.000000
25%     18.500000   20000.000000
50%     28.000000   50000.000000
75%     40.500000   60000.000000
max    240.000000  750000.000000


In [3]:
# Handle missing values
imputer = SimpleImputer(strategy="most_frequent")
df[:] = imputer.fit_transform(df)

# Drop duplicates if any
df = df.drop_duplicates()

In [4]:
label_encoders = {}

for col in df.columns:
    if df[col].dtype == "object":
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

In [5]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        print(col, X_train[col].unique()[:5])

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesClassifier

# include both object + string (future-proof pandas)
categorical_cols = X_train.select_dtypes(include=['object', 'string']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', ExtraTreesClassifier(n_estimators=200, random_state=42))
])

model.fit(X_train, y_train)
preds = model.predict(X_test)

In [13]:
X_train.dtypes

Name        str
Type        str
Amount    int64
dtype: object